<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week8/Day4/Dailychallenges/Daily_Challenge_MCP_Airbnb_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student Notebook - MCP + Airbnb (Colab)

Reference notebook: local notes MCP + Airbnb MCP + optional real LLM.

## Install
Run once. npm only needed for the real Airbnb server.

In [31]:
!pip install -q mcp nest_asyncio requests
!pip install azure-ai-inference

# Optional: real Airbnb server
!npm install -g @openbnb/mcp-server-airbnb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.9/124.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 12.9 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙npm warn deprecated whatwg-encoding@3.1.1: Use @exodus/bytes instead for a more spec-conformant and faster implementation
⠙⠹⠸⠼npm warn deprecated node-domexception@1.0.0: Use your platform's native DOMException instead
⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 124 packages in 8s
⠇
⠇51 packages are looking for funding
⠇  run `npm fund` for details
⠇

In [32]:
import sys
from ipykernel.iostream import OutStream

def _patched_fileno(self):
    # stdout → 1, stderr → 2
    if self is sys.stderr:
        return 2
    return 1

# Patch the class for all OutStream instances
OutStream.fileno = _patched_fileno

# And patch the current instances explicitly
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2


## Config
Flip toggles as needed. Keep defaults for stubbed run.

In [38]:
import os
from google.colab import userdata

try:
    # On essaie de récupérer le token
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
    print("✅ GITHUB_TOKEN récupéré avec succès !")
except Exception:
    # Si la clé n'existe pas dans l'interface Colab, on ne bloque pas le code
    os.environ["GITHUB_TOKEN"] = ""
    print("⚠️ GITHUB_TOKEN non trouvé dans les Secrets du Colab (Ce n'est pas grave, le mode Stub sera utilisé).")

print("GITHUB_TOKEN visible to Python:", bool(os.getenv("GITHUB_TOKEN")))

⚠️ GITHUB_TOKEN non trouvé dans les Secrets du Colab (Ce n'est pas grave, le mode Stub sera utilisé).
GITHUB_TOKEN visible to Python: False


In [39]:
import os
BASE_ENV = os.environ.copy()
BASE_ENV["MCP_HTTP_TOKEN"] = MCP_HTTP_TOKEN


In [40]:
import os
from google.colab import userdata  # Colab secrets API

# If your secret is saved under the key "GITHUB_TOKEN" in Colab:
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

# If you used a different key name in the secrets UI, e.g. "github_token":
# os.environ["GITHUB_TOKEN"] = userdata.get("github_token")

print("GITHUB_TOKEN visible to Python:", bool(os.getenv("GITHUB_TOKEN")))


SecretNotFoundError: Secret GITHUB_TOKEN does not exist.

## Local notes MCP server

In [ ]:

LOCAL_SERVER = Path("local_notes_server.py")
LOCAL_SERVER.write_text(
'''from mcp.server.fastmcp import FastMCP
notes = []
mcp = FastMCP() # construct FastMCP server instance

@mcp.tool() # Add decorator to register list_notes as a tool
def add_note(text: str) -> str:
    'Add a note to the in-memory list.'
    notes.append(text)
    return f"Saved note #{len(notes)}: {text}"

@mcp.tool() # Add decorator to register list_notes as a tool
def list_notes() -> str:
    'List saved notes.'
    if not notes:
        return "No notes yet"
    return "".join(f"{i+1}. {n}" for i, n in enumerate(notes))

if __name__ == "__main__":
    mcp.run()
'''.strip() + "",
    encoding="utf-8",
)
print("wrote", LOCAL_SERVER)


## Client helpers (convert tools, stub planner, optional real LLM)

In [ ]:

import asyncio
import json
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def convert_tool(tool, prefix: str):
    # Azure requires ^[a-zA-Z0-9_\.-]+$, so no slashes
    fn_name = f"{prefix}__{tool.name}"
    return {
        "type": "function",
        "function": {
            "name": fn_name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }



def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    import os
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
    # To-Do: Read documentation (hover over) and finish the function call
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls


In [ ]:
def answer_with_llm(
    user_prompt: str,
    tool_calls: List[Dict[str, Any]],
    tool_results: List[Dict[str, Any]],
    use_real: bool = True,
) -> str:
    import os
    import json

    # MINIMAL FIX: shrink tool_results before sending to gpt-4o
    small_results = []
    for r in tool_results:
        content = r.get("content", [])
        short_content = []
        if content:
            first = content[0]
            if isinstance(first, str) and len(first) > 4000:
                first = first[:4000] + "...(truncated)..."
            short_content = [first]
        small_results.append(
            {
                "name": r.get("name"),
                "args": r.get("args", {}),
                "content": short_content,
            }
        )


    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use_real=False in answer_with_llm.")

    client = ChatCompletionsClient(
        "https://models.inference.ai.azure.com",
        AzureKeyCredential(token),
    )

    payload = {
        "user_question": user_prompt,
        "tool_calls": tool_calls,
        # use the shrunk version here
        "tool_results": small_results,
    }

    resp = client.complete(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You answer the user's question using the given tool outputs.\n"
                    "JSON contains user_question, tool_calls, and tool_results (already truncated).\n"
                    "1. Answer clearly in markdown.\n"
                    "2. At the end, add:\n"
                    "## Tools used\n"
                    "- One bullet per distinct tool name.\n"
                ),
            },
            {
                "role": "user",
                "content": json.dumps(payload, ensure_ascii=False),
            },
        ],
        temperature=0,
        max_tokens=600,
    )

    msg = resp.choices[0].message
    parts = getattr(msg, "content", None)
    if isinstance(parts, list):
        texts = []
        for p in parts:
            text = getattr(p, "text", None) or getattr(p, "content", None)
            if isinstance(text, str):
                texts.append(text)
        if texts:
            return "".join(texts)

    return str(msg.content)


## Orchestrate (connect both servers and execute tool_calls)

In [41]:
import asyncio
import os
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from google.colab import userdata

# Autoriser l'exécution d'asyncio dans l'environnement de boucle d'événements de Colab
nest_asyncio.apply()

# --- CONFIGURATION DU JETON ET DES CONSTANTES ---
# Récupération sécurisée du token depuis les secrets Colab
try:
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
except Exception:
    # Si le secret n'est pas configuré, on s'assure qu'il n'y ait pas de plantage brutal ici
    pass

# Variables de contrôle (à ajuster selon vos besoins)
USE_REAL_AIRBNB = False  # Passez à True si vous utilisez le vrai serveur via npx
USE_REAL_LLM = bool(os.getenv("GITHUB_TOKEN"))  # True si un GITHUB_TOKEN valide est détecté

async def orchestrate(prompt: str):
    # --- 1. CONFIGURATION DES PARAMÈTRES DU SERVEUR LOCAL (NOTES) ---
    # On cible le fichier du serveur de notes généré dans les cellules précédentes
    local_params = StdioServerParameters(
        command="python",
        args=["local_notes_server.py"],  # Doit correspondre au nom du fichier écrit via %%writefile
        env=os.environ.copy(),
    )

    # --- 2. CONFIGURATION DES PARAMÈTRES DU SERVEUR AIRBNB ---
    if USE_REAL_AIRBNB:
        print("Configuration du serveur Airbnb RÉEL (via npx)...")
        airbnb_params = StdioServerParameters(
            command="npx",
            args=["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"],
            env=os.environ.copy(),
        )
    else:
        print("Configuration du serveur STUB Airbnb (Simulé)...")
        airbnb_params = StdioServerParameters(
            command="python",
            args=["airbnb_stub_server.py"],  # Doit correspondre au fichier stub créé par le Colab
            env=os.environ.copy(),
        )

    # --- 3. CONNEXION ET ROUTAGE DES COMMUNICATIONS (STDIO) ---
    print(f"\n🚀 Démarrage de l'orchestrateur pour la requête : '{prompt}'")

    async with stdio_client(local_params) as local_transport, \
               stdio_client(airbnb_params) as airbnb_transport:

        async with ClientSession(local_transport[0], local_transport[1]) as local_session, \
                   ClientSession(airbnb_transport[0], airbnb_transport[1]) as airbnb_session:

            # Initialisation protocolaire obligatoire pour le protocole MCP
            await local_session.initialize()
            await airbnb_session.initialize()
            print("✅ Sessions MCP connectées avec succès !")

            # --- 4. EXÉCUTION DU SCÉNARIO D'AGENCE ---
            # Étape A : Recherche d'hébergement sur Airbnb
            print("🔍 Appel de l'outil 'airbnb_search'...")
            search_query = {"location": "Paris"}  # Paramètre par défaut pour le stub
            if USE_REAL_AIRBNB:
                search_query = {"location": "Paris", "checkin": "2026-08-01"}

            search_response = await airbnb_session.call_tool("airbnb_search", search_query)
            listings_text = search_response.content[0].text
            print("✨ Résultats de recherche récupérés avec succès.")

            # Étape B : Enregistrement des résultats sous forme de note locale
            print("📝 Appel de l'outil 'add_note' pour sauvegarder les résultats...")
            note_payload = {
                "note": f"Résultats de recherche pour {prompt} :\n{listings_text[:250]}..."
            }
            await local_session.call_tool("add_note", note_payload)
            print("✅ Note ajoutée au serveur local.")

            # Étape C : Vérification finale et affichage des notes enregistrées
            print("📋 Récupération de la liste des notes mises à jour...")
            notes_list_response = await local_session.call_tool("list_notes", {})

            print("\n================ ÉTAT FINAL DES NOTES ================")
            print(notes_list_response.content[0].text)
            print("=======================================================")

            return "Succès"

# --- LANCEMENT DE LA DÉMO ---
# Requête test pour valider le fonctionnement de bout en bout
await orchestrate("Trouver un logement à Paris pour mon week-end")

Configuration du serveur STUB Airbnb (Simulé)...

🚀 Démarrage de l'orchestrateur pour la requête : 'Trouver un logement à Paris pour mon week-end'


ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)

## Demo
Adjust the prompt as you like. Switch `USE_REAL_AIRBNB/USE_REAL_LLM` to true when ready.

In [ ]:
from pathlib import Path
import os
import sys
from ipykernel.iostream import OutStream
import io

prompt = "What tools can you access ? list them please "

# Ensure LOCAL_SERVER is defined for this execution context
LOCAL_SERVER = Path("local_notes_server.py")

# Define MCP_HTTP_TOKEN, USE_REAL_LLM, and USE_REAL_AIRBNB for this execution context
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = True # Assuming this should be True based on notebook config
USE_REAL_AIRBNB = True # Assuming this should be True based on notebook config

# Define BASE_ENV
BASE_ENV = os.environ.copy()
BASE_ENV["MCP_HTTP_TOKEN"] = MCP_HTTP_TOKEN

# --- Patch for UnsupportedOperation: fileno issue ---
def _patched_fileno(self):
    if self is sys.stderr:
        return 2
    return 1

# Patch the class for all OutStream instances
OutStream.fileno = _patched_fileno

# And patch the current instances explicitly
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2
# --- End patch ---

# Call orchestrate and display results
tool_calls, tool_results = await orchestrate(prompt)
print("tool_calls:", tool_calls)

# Answer using the LLM
answer = answer_with_llm(prompt, tool_calls, tool_results, use_real=USE_REAL_LLM)
print(answer)